### Imports

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, classification_report,
    f1_score, precision_score, recall_score, accuracy_score, roc_auc_score,
    roc_curve, ConfusionMatrixDisplay
)

In [2]:
column_names = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital_status', 
                'occupation', 'relationship', 'race', 'sex', 'capital-gain', 
                'capital-loss', 'hours-per-week', 'native-country', 'income']

adult = pd.read_csv('adult.data',
           skipinitialspace=True,
           na_values='?',
           header=None,
           names=column_names)

In [3]:
data = pd.read_csv('adult.data', header=None, names=column_names)
adult.head()

,age,workclass,fnlwgt,education,education-num,marital_status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


### Data inspection

In [4]:
print(adult.head, '\n\n\n', adult.info(), '\n\n\n', adult.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       30725 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education-num   32561 non-null  int64 
 5   marital_status  32561 non-null  object
 6   occupation      30718 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital-gain    32561 non-null  int64 
 11  capital-loss    32561 non-null  int64 
 12  hours-per-week  32561 non-null  int64 
 13  native-country  31978 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB
<bound method NDFrame.head of        age         workclass  fnlwgt   education  education-num

In [5]:
adult['income'].unique()

array(['<=50K', '>50K'], dtype=object)

In [6]:
print(f"\nDataset shape: {adult.shape}")
print(f"Missing values:\n{adult.isnull().sum()}")

# Clean target — strip trailing periods, map to binary
adult["income"] = adult["income"].str.strip()
adult["income"] = adult["income"].map({"<=50K": 0, ">50K": 1})

print(f"\nTarget distribution:\n{adult['income'].value_counts()}")
print(f"Class imbalance — '>50K' share: {adult['income'].mean():.2%}")


Dataset shape: (32561, 15)
Missing values:
age                  0
workclass         1836
fnlwgt               0
education            0
education-num        0
marital_status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital-gain         0
capital-loss         0
hours-per-week       0
native-country     583
income               0
dtype: int64

Target distribution:
income
0    24720
1     7841
Name: count, dtype: int64
Class imbalance — '>50K' share: 24.08%


### Data Splitting

In [7]:
X = adult.drop("income", axis=1)
y = adult["income"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y          # ← critical for imbalanced classes
)

print(f"Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")
print(f"Train >50K rate: {y_train.mean():.2%} | Test >50K rate: {y_test.mean():.2%}")

Train size: 26048 | Test size: 6513
Train >50K rate: 24.08% | Test >50K rate: 24.07%


### Preprocessing Pipeline

In [8]:
NUMERICAL_COLUMNS = ["age", "fnlwgt", "education-num", "capital-gain", "capital-loss", "hours-per-week"]
CATEGORICAL_COLUMNS = ["workclass", "education", "marital_status", "occupation",
               "relationship", "race", "sex", "native-country"]


# Numerical: median imputation → StandardScaler
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler())
])

# Categorical: mode imputation → OneHotEncoder
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe",     OneHotEncoder(drop='first', handle_unknown="ignore", sparse_output=False))
])

# Combine into ColumnTransformer
preprocessor = ColumnTransformer([
    ("num", num_pipeline, NUMERICAL_COLUMNS),
    ("cat", cat_pipeline, CATEGORICAL_COLUMNS)
])

## Hyperparameter Tuning and Cross Validation

In [9]:
# skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# best_estimators = {}
# cv_results_summary = {}

# final_pipe = Pipeline([
#     ('preprocessor', preprocessor),
#     ('clf', DecisionTreeClassifier()),
#     # ('model_2', DecisionTreeClassifier())
# ])

# grid = {
#     'clf__max_depth': np.arange(10, 110, 10)
# }

# decision_tree = DecisionTreeClassifier()

# search = RandomizedSearchCV(
#     final_pipe,
#     grid,
#     n_iter=10,
#     cv=skfold,
#     scoring='f1',
#     random_state=42,
#     verbose=1
# )

### Defining the Models

In [10]:
MODELS = {
    "KNN": {
        "clf": KNeighborsClassifier(),
        "params": {
            "classifier__n_neighbors": [3, 5, 7, 9, 11, 15, 21, 31],
            "classifier__weights": ["uniform", "distance"],
            "classifier__p": [1, 2]
        }
    },
    
    "DecisionTree": {
        "clf": DecisionTreeClassifier(),
        "params": {
            "classifier__criterion": ["gini", "entropy", "log_loss"],
            "classifier__max_depth": [None, 5, 10, 15, 20, 30, 50],
            "classifier__min_samples_split": [2, 5, 10, 20, 50],
            "classifier__min_samples_leaf": [1, 2, 4, 8, 16],
            "classifier__max_features": [None, "sqrt", "log2"],
            "classifier__ccp_alpha": [0.0, 0.0001, 0.001, 0.01]
        }
    },
    
    "RandomForest": {
        "clf": RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1),
        "params": {
            "classifier__n_estimators": [100, 200],
            "classifier__max_depth": [None, 10, 20],
            "classifier__min_samples_split": [2, 5],
            "classifier__max_features": ["sqrt", "log2"]
        }
    }   
    
}
    
    # "GradientBoosting": {
    #     "clf": K(random_state=42),
    #     "params": {
    #         "classifier__n_estimators": [100, 200],
    #         "classifier__max_depth": [3, 5],
    #         "classifier__learning_rate": [0.05, 0.1],
    #         "classifier__subsample": [0.8, 1.0]
    #     }
    # },
    # "LogisticRegression": {
    #     "clf": LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
    #     "params": {
    #         "classifier__C": [0.01, 0.1, 1.0, 10.0],
    #         "classifier__solver": ["lbfgs", "saga"],
    #         "classifier__penalty": ["l2"]
#         }
#     }
# }

### Cross Validation

We go through each model list in the MODELS `(DICT)` and pass it through a pipeline containing our preprocessor to clean the data each them and then, use the classifier object listed in the dict as well

In [50]:
for name, config in MODELS.items():
    print(f"\n  Tuning {name}...")

    full_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier",   config["clf"])
    ])

    # HyperParameter Tuning using RandomSearch
    search = RandomizedSearchCV(
        full_pipeline,
        param_distributions=config["params"],
        n_iter=12,
        scoring="f1",
        cv=cv_strategy,
        random_state=42,
        n_jobs=-1,
        verbose=1
    )
    search.fit(X_train, y_train)
    best_estimators[name] = search.best_estimator_
    cv_results_summary[name] = {
        "best_cv_f1": search.best_score_,
        "best_params": search.best_params_
    }
    print(f"    Best CV F1-score: {search.best_score_:.4f}")
    print(f"    Best params: {search.best_params_}")


  Tuning KNN...
Fitting 5 folds for each of 12 candidates, totalling 60 fits
    Best CV F1-score: 0.6405
    Best params: {'classifier__weights': 'uniform', 'classifier__p': 2, 'classifier__n_neighbors': 31}

  Tuning DecisionTree...
Fitting 5 folds for each of 12 candidates, totalling 60 fits
    Best CV F1-score: 0.6452
    Best params: {'classifier__min_samples_split': 10, 'classifier__min_samples_leaf': 8, 'classifier__max_features': None, 'classifier__max_depth': 50, 'classifier__criterion': 'log_loss', 'classifier__ccp_alpha': 0.0001}

  Tuning RandomForest...
Fitting 5 folds for each of 12 candidates, totalling 60 fits
    Best CV F1-score: 0.6961
    Best params: {'classifier__n_estimators': 100, 'classifier__min_samples_split': 2, 'classifier__max_features': 'sqrt', 'classifier__max_depth': 20}


## Model Evaluation

In [55]:
eval_results = {}

for name, estimator in best_estimators.items():
    y_pred  = estimator.predict(X_test)
    y_proba = estimator.predict_proba(X_test)[:, 1]

    cm     = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred, target_names=["<=50K", ">50K"])

    acc    = accuracy_score(y_test, y_pred)
    prec   = precision_score(y_test, y_pred)
    rec    = recall_score(y_test, y_pred)
    f1     = f1_score(y_test, y_pred)
    auc    = roc_auc_score(y_test, y_proba)
    fpr, tpr, _ = roc_curve(y_test, y_proba)

    eval_results[name] = {
        "cm": cm, "report": report,
        "accuracy": acc, "precision": prec,
        "recall": rec, "f1": f1, "auc": auc,
        "fpr": fpr, "tpr": tpr,
        "y_pred": y_pred, "y_proba": y_proba,
        "best_cv_f1": cv_results_summary[name]["best_cv_f1"]
    }